# 03. Feature Engineering Analysis

This notebook visualizes the engineered features and their relationships with the target electricity price.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.features import DataProcessor
from src.utils import load_data

sns.set_theme(style="whitegrid", palette="viridis")

# Load processed features
df = load_data('processed', 'se3_features_v1.parquet')
print(f"Loaded {len(df)} rows and {len(df.columns)} features.")

## 1. Feature Correlation Heatmap

Understand which features are most strongly related to the target price and identify potential multi-collinearity.

In [ ]:
plt.figure(figsize=(15, 10))
corr = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title("Feature Correlation Matrix", fontsize=15)
plt.show()

## 2. Target vs. Key Features

Visualize relationship with Rolling Averages and Lags.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 24h Rolling Mean
sns.scatterplot(data=df.sample(2000), x='value_rolling_mean_24h', y='value', alpha=0.3, ax=axes[0, 0])
axes[0, 0].set_title("Price vs. 24h Rolling Mean")

# Lag 24h
sns.scatterplot(data=df.sample(2000), x='value_lag_24', y='value', alpha=0.3, ax=axes[0, 1])
axes[0, 1].set_title("Price vs. 24h Lag")

# Hour of day distribution
sns.boxplot(data=df, x='hour', y='value', ax=axes[1, 0], palette="flare")
axes[1, 0].set_title("Price Distribution by Hour")

# Is Peak Morning vs Standard
sns.violinplot(data=df, x='is_peak_morning', y='value', ax=axes[1, 1])
axes[1, 1].set_title("Price: Peak Morning vs. Other Hours")

plt.tight_layout()
plt.show()

## 3. Distribution of New Features

Check for skewness or outliers in engineered features like Volatility.

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df['value_rolling_std_24h'], kde=True, color='purple')
plt.title("Distribution of 24h Price Volatility (Rolling Std)")
plt.show()

Advanced Feature Engineering (Lags & Rolling Stats)
Date: March 5, 2026
Module: 
src/data_processor.py
 & 
src/features.py

Lag Features (Time-Series Inertia)
Captured historical dependencies to help the model learn from past price behaviors:

value_lag_1: 1-hour lookback to capture short-term autocorrelation (market inertia).
value_lag_24: 24-hour lookback to capture daily periodicity (Yesterday's price level).
value_lag_168: 168-hour (1 week) lookback to capture weekly seasonality (Weekend vs. Weekday patterns).
Rolling Window Statistics (24h Window)
Implemented statistical windows to quantify market trends and volatility:

value_rolling_mean_24h: Identifies the current price baseline by smoothing fluctuations.
value_rolling_max_24h / value_rolling_min_24h: Defines the dynamic boundaries (Price Envelope) of the market.
value_rolling_std_24h: A volatility index used to quantify market uncertainty.
value_diff_24h: Momentum indicator tracking price shifts relative to 24 hours ago.

In [ ]:
# New Visualizations for Lags and Rolling Windows
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Autocorrelation: Current vs 1h Lag
sns.regplot(data=df.sample(2000), x='value_lag_1', y='value', 
            scatter_kws={'alpha':0.2}, line_kws={'color':'red'}, ax=axes[0, 0])
axes[0, 0].set_title("Price vs. 1h Lag (Short-term Continuity)")

# 2. Daily Seasonality: Current vs 24h Lag
sns.regplot(data=df.sample(2000), x='value_lag_24', y='value', 
            scatter_kws={'alpha':0.2}, line_kws={'color':'green'}, ax=axes[0, 1])
axes[0, 1].set_title("Price vs. 24h Lag (Daily Pattern)")

# 3. Rolling Envelope (Last 7 days)
sample_week = df.iloc[-168:]
axes[1, 0].plot(sample_week['timestamp'], sample_week['value'], label='Actual', color='black')
axes[1, 0].fill_between(sample_week['timestamp'], 
                         sample_week['value_rolling_min_24h'], 
                         sample_week['value_rolling_max_24h'], 
                         color='orange', alpha=0.3, label='24h Range')
axes[1, 0].set_title("Price with 24h Min-Max Envelope")
axes[1, 0].legend()

# 4. Volatility Distribution
sns.histplot(df['value_rolling_std_24h'], kde=True, ax=axes[1, 1], color='purple')
axes[1, 1].set_title("24h Price Volatility Distribution (Rolling Std)")

plt.tight_layout()
plt.show()
